# Part A · Reproduction with the authors' code (Kaggle)

Each section runs one stage of the authors' code in `original_code/`. The notebook only adds what Kaggle needs: data on local disk, missing config files, and outputs kept in `/kaggle/working`.

**Notebook settings:** Accelerator *GPU T4 ×2*, Internet *on* (needs a phone-verified account). Add the secrets `GH_TOKEN` and `HF_TOKEN` under *Add-ons → Secrets*.

**Saved outputs (`/kaggle/working/`):** `checkpoints/` (weights and temperatures), `y-prediction/` (logits), `evaluation/` (metrics), `configs/`. Kaggle keeps these only when you **Save Version** (*Save & Run All* runs in the background for up to 12 h; the output cap is 20 GB). To continue later, add that version's output as an input (*Add Input → Your Work → Notebooks*) and set `PREVIOUS_OUTPUT` to its path under `/kaggle/input/`.

**Each session:** run sections 0–2, then the stage you need. Every stage skips work whose outputs already exist.

In [ ]:
from pathlib import Path
import torch
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
OUT = Path('/kaggle/working')
PREVIOUS_OUTPUT = None              # e.g. Path('/kaggle/input/<this-notebook>') to resume from an earlier version
N_GPUS = torch.cuda.device_count()  # 2 on T4 x2

REPO_URL = "https://github.com/gorrast/DD2610-Final-Project.git"
GITHUB_TOKEN_SECRET = "GH_TOKEN"
HF_TOKEN_SECRET = "HF_TOKEN"

## 0 · Code and environment

In [ ]:
import os, sys, shutil
from base64 import b64encode

# The repo lives in /tmp so Ray's trial checkpoints (ray_results/) are not saved as notebook output.
auth = b64encode(f"x-access-token:{secrets.get_secret(GITHUB_TOKEN_SECRET)}".encode()).decode()
!rm -rf /tmp/repo && git -c http.extraHeader="Authorization: Basic {auth}" clone -q --depth 1 -b partA-kaggle {REPO_URL} /tmp/repo
%cd /tmp/repo/original_code
if PREVIOUS_OUTPUT:
    shutil.copytree(PREVIOUS_OUTPUT, OUT, dirs_exist_ok=True)
for d in ['checkpoints', 'y-prediction', 'evaluation', 'configs']:
    (OUT / d).mkdir(parents=True, exist_ok=True)
    os.symlink(OUT / d, d)

# Scripts in train/ import the package folders, so they need original_code on the path.
sys.path.insert(0, os.getcwd())
os.environ.update(PYTHONPATH=os.getcwd(), WANDB_MODE='disabled', PYTHONUNBUFFERED='1')
!pip install -q wilds timm "ray[tune]" fvcore

## 1 · Data
Downloads go to `/tmp/data` on local disk, outside `/kaggle/working`, so they don't count toward the 20 GB output. The authors' `ImageNetDataset` reads `ImageFolder(root/val)` and `ImageFolder(root/train)`. The ImageNet images are therefore written to class folders `000`…`999`, which sort into the standard label order. `train` points to `val` because we only evaluate pretrained ImageNet models. The parquet shards are deleted after extraction to save disk.

In [ ]:
from torchvision.datasets import Caltech256
from torchvision.datasets.utils import download_and_extract_archive
from huggingface_hub import snapshot_download
import pyarrow.parquet as pq

DATA = Path('/tmp/data')
IMAGENET, IMAGENET_V2 = DATA / 'imagenet', DATA / 'imagenetv2-matched-frequency-format-val'

Caltech256(DATA, download=True)
download_and_extract_archive('https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/'
                             'imagenetv2-matched-frequency.tar.gz', DATA, filename='imagenetv2.tar', remove_finished=True)
snapshot_download('ILSVRC/imagenet-1k', repo_type='dataset', local_dir=DATA / 'hf',
                  allow_patterns=['data/validation-*.parquet'], token=secrets.get_secret(HF_TOKEN_SECRET))

for shard in sorted((DATA / 'hf/data').glob('validation-*.parquet')):
    for row in pq.read_table(shard).to_pylist():
        out = IMAGENET / 'val' / f"{row['label']:03d}" / Path(row['image']['path']).name
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_bytes(row['image']['bytes'])
(IMAGENET / 'train').symlink_to('val')
shutil.rmtree(DATA / 'hf')

In [ ]:
# Check the seed-42 splits with the authors' loaders.
from load_data import get_dataloaders
from load_models.transforms import get_transforms

caltech = get_dataloaders('caltech256', DATA, 64, 2, get_transforms())
imagenet = get_dataloaders('imagenet', IMAGENET, 64, 2, get_transforms(), ood_root_dir=IMAGENET_V2)
print({k: len(v.dataset) for k, v in caltech.items() if v})   # train 21424, val 4591, test 4592
print({k: len(v.dataset) for k, v in imagenet.items() if v})  # train 50000 (= val), val 2500, test 47500, ood_test 10000

## 2 · Configs (missing from the release) and FLOPs
* **`backbones.csv`:** the 8 models from the README. `4_evaluate_predictions.py` also reads `GFLOPS` and `Params` from this file.
* **`duos.csv`:** 2 bases × 6 sidekicks with `gflops_*`. The FLOP count (step 8) happens here because `5_evaluate_duo_predictions.py` reads the GFLOPs from this file. We use fvcore on one 224×224 image with fused attention off. fvcore prints a warning for any operator it can't count.
* **`caltech256.yaml`:** batch size 128, 8 LP epochs and 16 FT epochs come from paper Appendix A.2. `warmup_epochs` and `grace_period` are **our choice**, because the paper doesn't give them. `num_workers` is split between the GPUs, because two trials share Kaggle's 4 CPUs.

In [ ]:
import pandas as pd, torch, timm, yaml
from fvcore.nn import FlopCountAnalysis
from load_models import get_model_with_head

timm.layers.set_fused_attn(False)
backbones = pd.DataFrame([
    ('convnext_base', 'torchvision', 'IMAGENET1K_V1'),
    ('vit_base_patch16_clip_224.laion2b_ft_in12k_in1k', 'timm', None),
    ('resnet10t.c3_in1k', 'timm', None),
    ('resnet18', 'torchvision', 'IMAGENET1K_V1'),   # ResNet weights: our choice
    ('resnet50', 'torchvision', 'IMAGENET1K_V1'),
    ('resnet101', 'torchvision', 'IMAGENET1K_V1'),
    ('resnet152', 'torchvision', 'IMAGENET1K_V1'),
    ('resnet200d.ra2_in1k', 'timm', None),
], columns=['model_name', 'source', 'tv_weights'])
BASES = backbones.model_name[:2].tolist()

for i, r in backbones.iterrows():
    model = get_model_with_head(r.model_name, None, r.source, r.tv_weights, keep_imagenet_head=True)[0].eval()
    backbones.loc[i, 'GFLOPS'] = FlopCountAnalysis(model, torch.randn(1, 3, 224, 224)).total() / 1e9
    backbones.loc[i, 'Params'] = sum(p.numel() for p in model.parameters()) / 1e6
backbones.to_csv('configs/backbones.csv', index=False)

g = backbones.set_index('model_name').GFLOPS
pd.DataFrame([(l, s, g[l], g[s], g[s] / g[l]) for l in BASES for s in g.index if s not in BASES],
             columns=['model_large', 'model_small', 'gflops_large', 'gflops_small', 'gflops_balance']
             ).to_csv('configs/duos.csv', index=False)
yaml.safe_dump({'training': {'batch_size': 128, 'num_epochs_lp': 8, 'num_epochs_ff': 16,
                             'num_workers': os.cpu_count() // N_GPUS, 'warmup_epochs': 2, 'grace_period': 4}},
               open('configs/caltech256.yaml', 'w'))
backbones

## 3 · ImageNet checkpoints  →  `train/download_imagenet_models.py`

In [ ]:
!python train/download_imagenet_models.py --backbone_csv configs/backbones.csv --model_save_dir checkpoints/imagenet

## 4 · Caltech-256 LP → FT  →  `train/hp_tune.py`
**Our reduced protocol** (fixed before looking at any test results): we tune only the learning rate.
* **LP:** 4 trials, learning rate in [1e-4, 1e-2], weight decay 0 (unchanged from the authors).
* **FT:** 4 trials (authors: 12), learning rate in [1e-6, 3e-4], weight decay **fixed at 1e-6** (authors searched [1e-8, 1e-5]). The fixed value is set in `hp_tune.py`.

The script requests 4 CPUs and 1 GPU per Ray trial. We start Ray with 4 logical CPUs per GPU, so on T4 ×2 two trials run at once. The script skips LP for a model whose LP checkpoint already exists, but always reruns FT, so list only the models still to train in `MODELS`. A session ends after 12 h, so split the 8 models over several saved versions.

**Who trains what.** `resnet18` is done. Hugo trains the two bases (`convnext_base`, CLIP ViT-B/16). Erik and Nina split the remaining ResNets in `ASSIGNED`, with about the same total GFLOPs each. Set `ME` to your name before running.

At the end, everyone's `checkpoints/` go into one output for steps 5–7: add each person's saved output as an input and copy them into `/kaggle/working`, as `PREVIOUS_OUTPUT` does.

In [ ]:
import ray

FT_TRIALS = 4                       # reduced protocol (authors: 12); LP uses 4 trials, set in hp_tune.py
ASSIGNED = {                        # resnet18 done; bases are Hugo's
    'Erik': ['resnet200d.ra2_in1k', 'resnet50', 'resnet10t.c3_in1k'],
    'Nina': ['resnet152', 'resnet101'],
}
ME = 'Erik'
MODELS = ASSIGNED[ME]               # one model per saved version if a session would pass 12 h

!ray stop > /dev/null 2>&1
ray.init(address='local', num_cpus=4 * N_GPUS, num_gpus=N_GPUS, include_dashboard=False)
os.environ.update(RAY_ADDRESS='auto', PYTHONUNBUFFERED='1')   # hp_tune.py connects to this Ray
for r in backbones[backbones.model_name.isin(MODELS)].itertuples():
    !python train/hp_tune.py --model_name {r.model_name} --source {r.source} --tv_weights {r.tv_weights} \
        --dataset_name caltech256 --dataset_dir {DATA} --num_samples {FT_TRIALS}

## 5 · Logits  →  `evaluate/1_single_model_prediction.py`
Writes `y-prediction/<dataset>/{val,test,ood_test}/raw/<model>.csv`. Before any models are combined, the accuracy check confirms that each model's output indices match the dataset labels.

In [ ]:
!python evaluate/1_single_model_prediction.py --dataset_name imagenet --dataset_dir {IMAGENET} --ood_dataset_dir {IMAGENET_V2} \
    --model_dir_path checkpoints/imagenet --backbone_csv configs/backbones.csv
!python evaluate/1_single_model_prediction.py --dataset_name caltech256 --dataset_dir {DATA} \
    --model_dir_path checkpoints/caltech256/ff --backbone_csv configs/backbones.csv

for ds in ['imagenet', 'caltech256']:
    pp = pd.read_csv(f'y-prediction/{ds}/test/point_prediction.csv')
    print(ds, pp.drop(columns='target').eq(pp.target, axis=0).mean().round(3).to_dict())

## 6 · Temperature scaling  →  `evaluate/2_temp_scale.py`, `evaluate/3_duo_temp_scale.py`
Fits on the validation logits. `2_` fits one temperature per model, giving the temperature-scaled base. `3_` fits the two Duo weights, giving the learned-weight Duo. ImageNetV2 reuses the ImageNet values.

In [ ]:
for ds in ['imagenet', 'caltech256']:
    !python evaluate/2_temp_scale.py --dataset_name {ds} --prediction_dir_path y-prediction/{ds}/val/raw
    !python evaluate/3_duo_temp_scale.py --dataset_name {ds} --prediction_dir_path y-prediction/{ds}/val/raw --duo_csv_path configs/duos.csv

## 7 · Evaluation  →  `evaluate/4_evaluate_predictions.py`, `evaluate/5_evaluate_duo_predictions.py`
How `mode` in the output maps to our comparisons:
* `logit_average` → equal-weight Duo
* `temperature_weighted` → learned-weight Duo
* `dictatorial_weighteduncertainty` → UQ-only

The temperature-scaled base comes from `4_`. Metrics: accuracy, AUROC, AURC, SAC@98.

In [ ]:
for ds in ['imagenet', 'caltech256']:
    !python evaluate/4_evaluate_predictions.py --dataset_name {ds} --backbone_csv configs/backbones.csv --prediction_dir_path y-prediction/{ds}
    !python evaluate/5_evaluate_duo_predictions.py --dataset_name {ds} --duo_csv_path configs/duos.csv --prediction_dir_path y-prediction/{ds}

## 8 · FLOP balance
Computed in step 2: `gflops_balance` = sidekick GFLOPs ÷ base GFLOPs in `configs/duos.csv`.

## 9 · Evaluator discrepancy (to do)
In `evaluate/5_evaluate_duo_predictions.py` (around lines 111 and 128), the `temperature_weighted` Duo reuses the UQ-only confidence, which is the Duo probability at the *base* model's prediction. Paper §3.1 scores the Duo's confidence on its *own* predicted class. We report both, with the paper-definition results clearly labelled.

## 10 · Figures (to do)
Plot the metrics against `gflops_balance` from `evaluation/eval_res/` to reproduce Figures 3–6, starting from `plot_notebooks/plot_2_duos.ipynb`. Compare Caltech-256 with Appendix G Figures 11 and 14.